### Read in phenotype manifest + data cleaning

In [5]:
import pandas as pd
import json

In [6]:
phenotype_manifest = pd.read_csv("phenotype_manifest.csv", usecols=[
    'description',
    'in_max_independent_set',
    'phenocode',
    'filename'
    ])

In [3]:
phenotype_manifest['description'] = phenotype_manifest['description'].fillna(phenotype_manifest['phenocode'])

### Read in excluded phenotypes

In [7]:
exclude_phenotypes = []

with open("exclude_phenotypes.txt", "r") as file:
    for line in file:
        if line[0]=='#':
            continue
        exclude_phenotypes.append(int(line.strip().split()[0]))

In [8]:
exclude_phenotypes

[113, 3553]

### Create filtering (maximally independent traits)

In [4]:
max_ind_set = phenotype_manifest[phenotype_manifest['in_max_independent_set']]

### Create filtering (ICD10 diseases)

In [9]:
with open("icd10_indices_n100.json", "r") as f:
    icd_indices = json.load(f)

In [10]:
icd_phenotypes = phenotype_manifest.iloc[icd_indices]

In [ ]:
icd_phenotypes.head(20)

### [DEPRECATED] Create .csv files from the track lists (for the deprecated lists)

In [ ]:
file_path = "DEPRECATED_track_lists_o3.json"
with open(file_path, "r") as json_file:
    depr_loaded_data = json.load(json_file)

In [ ]:
depr_file_name_dict = {}

In [ ]:
for key, value in depr_loaded_data.items():
    # Convert list to DataFrame
    df_temp = pd.DataFrame(value)

    # Create file name:
    if (not isinstance(key, str)):
        raise KeyError(f"expected description string as identifier but got {type(key)}")
    else:
        temp_file_name = "_".join(key.split()[:4])
        temp_file_name = temp_file_name.replace("(", "").replace(")", "").replace("/", "").replace("|", "").replace("-", "").replace("[", "").replace("]", "")
        temp_file_name = f"DEPRECATED_tracklists/{temp_file_name}.csv"
        depr_file_name_dict[key] = temp_file_name

    # Save to CSV without a header or index
    df_temp.to_csv(temp_file_name, index=False, header=False)

### Create .csv files from the track lists

In [12]:
file_path = "track_lists_o3.json"
with open(file_path, "r") as json_file:
    loaded_data = json.load(json_file)

In [ ]:
k=0
for key, value in loaded_data.items():
    # For testing purposes: only generate 5 phenotype csv's
    if k>5:
        continue

    # Skip excluded phenotypes
    if int(key) in exclude_phenotypes:
        print(f"Skipping excluded phenotype {key}.")
        continue

    # only generate csv's for relevant indices:
    if int(key) not in icd_indices:
        continue

    # Convert list to DataFrame
    df_temp = pd.DataFrame(value)

    # Create file name:
    if (not isinstance(key, str)):
        raise KeyError(f"expected phenotype manifest index as string but got {type(key)}")
    else:
        csv_file_name = "tracklists/tracks_phen" + key + ".csv"

    # Save to CSV without a header or index
    df_temp.to_csv(csv_file_name, index=False, header=False)

    k+=1

### Write .txt file (for maximally independent set)

The sumstats files are saved like this:

`/cluster/customapps/biomed/boeva/lrabuzin/continuous-103220-both_sexes.tsv.bgz`

The 1KG files are saved like this:

`/cluster/work/boeva/lrabuzin/1000genomes_as_csv`

The exon region files are saved like this: #TODO

In [ ]:
with open("method_parameters_independent_set.txt", "w") as file:
    for index, row in max_ind_set.iterrows():
        if index in exclude_phenotypes:
            print(f"Skipping excluded phenotype {index} {row['description']}.")
            continue
        file.write(f"""/cluster/customapps/biomed/boeva/lrabuzin/ukbb_phens/downloads/{row['filename']} 
                   /cluster/work/boeva/lrabuzin/1000genomes_as_csv
                   /cluster/work/boeva/lrabuzin/genome_assembly/exon_regions_v2.csv 
                   /cluster/work/boeva/lrabuzin/{file_name_dict[row['description']]}\n""")

### Write .txt file (for ICD10 diseases)

The sumstats files are saved like this:

`/cluster/customapps/biomed/boeva/lrabuzin/continuous-103220-both_sexes.tsv.bgz`

The 1KG files are saved like this:

`/cluster/work/boeva/lrabuzin/1000genomes_as_csv`

The exon region files are saved like this: #TODO

In [17]:
with open("method_icd_phenotypes.txt", "w") as file:
    for index, row in icd_phenotypes.iterrows():
        if index in exclude_phenotypes:
            print(f"Skipping excluded phenotype {index} {row['description']}.")
            continue
        file.write(f"""/cluster/customapps/biomed/boeva/lrabuzin/ukbb_phens/downloads/{row['filename']} 
                   /cluster/work/boeva/lrabuzin/1000genomes_as_csv
                   /cluster/work/boeva/lrabuzin/genome_assembly/exon_regions_v2.csv 
                   /cluster/work/boeva/lrabuzin/{"tracklists/tracks_phen" + str(index) + ".csv"}\n""")